<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/AI_Ecosystem_Network_Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ecosystem Network Explorer
This notebook visualizes the global artificial intelligence ecosystem as a dynamic network, mapping the relationships between companies, models, researchers, and infrastructure.

In [1]:
!pip install pyvis networkx pandas numpy plotly sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.8 MB/s eta 0:00:00


In [4]:
import networkx as nx
import pandas as pd
import numpy as np
from pyvis.network import Network
import plotly.graph_objects as go
import json

class AIEcosystemGraph:
    def __init__(self):
        self.G = nx.DiGraph()

    def add_node(self, node_id, label, category, **kwargs):
        # Ensure category isn't duplicated in kwargs
        metadata = kwargs.copy()
        metadata['category'] = category
        metadata['label'] = label
        self.G.add_node(node_id, **metadata)

    def add_edge(self, source, target, relationship, weight=1.0, **kwargs):
        self.G.add_edge(source, target, title=relationship, weight=weight, **kwargs)

# Initialize the explorer
explorer = AIEcosystemGraph()

# Seed Data: Companies (Fixed logic to prevent multiple values for 'category')
companies = [
    ('OpenAI', 'AI Company', {'valuation': '80B', 'hq': 'San Francisco'}),
    ('Anthropic', 'AI Company', {'valuation': '18B', 'hq': 'San Francisco'}),
    ('NVIDIA', 'Infrastructure', {'valuation': '2T', 'hq': 'Santa Clara'}),
    ('Microsoft', 'Investor/Cloud', {'hq': 'Redmond'}),
    ('Google DeepMind', 'AI Company', {'hq': 'London'}),
    ('Meta AI', 'AI Company', {'hq': 'Menlo Park'}),
    ('TSMC', 'Infrastructure', {'type': 'Semiconductor'}),
    ('AWS', 'Infrastructure', {'type': 'Cloud Provider'})
]

for name, cat, attrs in companies:
    explorer.add_node(name, name, cat, **attrs)

# Seed Data: Edges
relationships = [
    ('Microsoft', 'OpenAI', 'Investment', 10.0),
    ('Microsoft', 'OpenAI', 'Partnership', 8.0),
    ('OpenAI', 'Microsoft', 'Cloud Dependency', 7.0),
    ('NVIDIA', 'TSMC', 'Manufacturing', 9.0),
    ('OpenAI', 'NVIDIA', 'Hardware Dependency', 9.0),
    ('Anthropic', 'AWS', 'Investment/Cloud', 7.0),
    ('Google DeepMind', 'Microsoft', 'Competitor', 5.0)
]

for u, v, rel, w in relationships:
    explorer.add_edge(u, v, rel, weight=w)

print(f'Graph fixed and initialized with {explorer.G.number_of_nodes()} nodes and {explorer.G.number_of_edges()} edges.')

Graph fixed and initialized with 8 nodes and 6 edges.


In [5]:
from IPython.display import HTML
import os

def visualize_basic_network(eg):
    # Initialize pyvis with notebook=True for Colab environments
    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white', directed=True, notebook=True)

    color_map = {
        'AI Company': '#3498db',
        'Infrastructure': '#e74c3c',
        'Investor/Cloud': '#f1c40f',
        'Research': '#2ecc71'
    }

    for node, data in eg.G.nodes(data=True):
        color = color_map.get(data.get('category'), '#95a5a6')
        net.add_node(node, label=data.get('label'), color=color, title=str(data))

    for source, target, data in eg.G.edges(data=True):
        net.add_edge(source, target, value=data.get('weight'), title=data.get('title'))

    net.toggle_physics(True)
    # Generate the HTML file
    net.show('ai_ecosystem.html')
    # Explicitly display the saved file
    if os.path.exists('ai_ecosystem.html'):
        return HTML(filename='ai_ecosystem.html')
    else:
        return "Error: HTML file not generated."

visualize_basic_network(explorer)

ai_ecosystem.html


## Community Detection and 3D Visualization
In this section, we use the Louvain community detection algorithm to find clusters in our graph and build a 3D interactive view using Plotly.

In [8]:
import community.community_louvain as louvain

# Perform Louvain community detection
# Note: Louvain requires an undirected graph
undirected_G = explorer.G.to_undirected()
partition = louvain.best_partition(undirected_G, weight='weight')

# Add community data to nodes
for node, community_id in partition.items():
    explorer.G.nodes[node]['community'] = community_id

print(f"Detected {len(set(partition.values()))} communities in the AI ecosystem.")

Detected 7 communities in the AI ecosystem.


In [9]:
def plot_3d_universe(eg):
    pos = nx.spring_layout(eg.G, dim=3, seed=42)

    edge_x, edge_y, edge_z = [], [], []
    for edge in eg.G.edges():
        x0, y0, z0 = pos[edge[0]]
        x1, y1, z1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    edge_trace = go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, line=dict(width=1, color='#888'), hoverinfo='none', mode='lines')

    node_x, node_y, node_z = [], [], []
    for node in eg.G.nodes():
        x, y, z = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_z.append(z)

    node_trace = go.Scatter3d(
        x=node_x, y=node_y, z=node_z, mode='markers+text',
        text=[eg.G.nodes[n].get('label', n) for n in eg.G.nodes()],
        marker=dict(showscale=True, colorscale='Viridis', size=8, color=[eg.G.nodes[n].get('community', 0) for n in eg.G.nodes()], line_width=2),
        hoverinfo='text'
    )

    fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(
        title='3D AI Ecosystem Universe',
        template='plotly_dark',
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False))
    ))
    fig.show()

plot_3d_universe(explorer)

## Interactive Dashboard and Export
Use the search bar below to find nodes semantically, and use the export function to save the graph data.

In [13]:
import ipywidgets as widgets
from IPython.display import clear_output

# Search Dashboard
search_input = widgets.Text(placeholder='e.g., companies building LLMs', description='Search AI:')
search_button = widgets.Button(description='Explore')
output = widgets.Output()

def on_search_clicked(b):
    with output:
        clear_output()
        query = search_input.value
        results = semantic_search(query, k=5)
        print(f"Results for: {query}\n")
        for node, desc in results:
            print(f"[ {node} ]")
            print(f"Details: {desc}\n")

search_button.on_click(on_search_clicked)
display(widgets.VBox([search_input, search_button, output]))

# Export Functions
def export_ecosystem():
    # Export as JSON for portability
    data = nx.node_link_data(explorer.G)
    with open('ai_ecosystem_graph.json', 'w') as f:
        json.dump(data, f)
    print("Ecosystem exported to ai_ecosystem_graph.json")

export_ecosystem()

Ecosystem exported to ai_ecosystem_graph.json


## AI-Powered Semantic Search
Using Sentence Transformers and FAISS, we enable natural language search across the ecosystem's nodes based on their metadata and descriptions.

In [12]:
from sentence_transformers import SentenceTransformer
import faiss

# 1. Prepare text descriptions for each node to build the vector space
node_list = list(explorer.G.nodes(data=True))
texts = []
for node_id, data in node_list:
    desc = f"{node_id} is a {data.get('category', 'entity')}. "
    desc += " ".join([f"{k}: {v}" for k, v in data.items() if k not in ['label', 'category', 'value', 'community']])
    texts.append(desc)

# 2. Generate Embeddings using a lightweight model
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts)

# 3. Create FAISS Index for similarity search
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

def semantic_search(query, k=3):
    query_vector = model.encode([query])
    distances, indices = index.search(np.array(query_vector).astype('float32'), k)

    results = []
    for i in range(k):
        idx = indices[0][i]
        node_id = node_list[idx][0]
        results.append((node_id, texts[idx]))
    return results

# Example Search to test the engine
query = "companies involved in high-end chip manufacturing"
search_results = semantic_search(query)
print(f"Top results for '{query}':")
for res in search_results:
    print(f"- {res[0]}: {res[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top results for 'companies involved in high-end chip manufacturing':
- OpenAI: OpenAI is a AI Company. valuation: 80B hq: San Francisco
- TSMC: TSMC is a Infrastructure. type: Semiconductor
- NVIDIA: NVIDIA is a Infrastructure. valuation: 2T hq: Santa Clara


## AI-Powered Semantic Search
Using Sentence Transformers and FAISS, we enable natural language search across the ecosystem's nodes based on their metadata and descriptions.

In [11]:
from sentence_transformers import SentenceTransformer
import faiss

# 1. Prepare text descriptions for each node
node_list = list(explorer.G.nodes(data=True))
texts = []
for node_id, data in node_list:
    desc = f"{node_id} is a {data.get('category', 'entity')}. "
    desc += " ".join([f"{k}: {v}" for k, v in data.items() if k not in ['label', 'category', 'value', 'community']])
    texts.append(desc)

# 2. Generate Embeddings
# Using a lightweight model for Colab efficiency
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts)

# 3. Create FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

def semantic_search(query, k=3):
    query_vector = model.encode([query])
    distances, indices = index.search(np.array(query_vector).astype('float32'), k)

    results = []
    for i in range(k):
        idx = indices[0][i]
        node_id = node_list[idx][0]
        results.append((node_id, texts[idx]))
    return results

# Example Search
query = "companies involved in high-end chip manufacturing"
search_results = semantic_search(query)
print(f"Top results for '{query}':")
for res in search_results:
    print(f"- {res[0]}: {res[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top results for 'companies involved in high-end chip manufacturing':
- OpenAI: OpenAI is a AI Company. valuation: 80B hq: San Francisco
- TSMC: TSMC is a Infrastructure. type: Semiconductor
- NVIDIA: NVIDIA is a Infrastructure. valuation: 2T hq: Santa Clara


## AI-Powered Semantic Search
Using Sentence Transformers and FAISS, we enable natural language search across the ecosystem's nodes based on their metadata and descriptions.

In [10]:
from sentence_transformers import SentenceTransformer
import faiss

# 1. Prepare text descriptions for each node
node_list = list(explorer.G.nodes(data=True))
texts = []
for node_id, data in node_list:
    desc = f"{node_id} is a {data.get('category', 'entity')}. "
    desc += " ".join([f"{k}: {v}" for k, v in data.items() if k not in ['label', 'category', 'value', 'community']])
    texts.append(desc)

# 2. Generate Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts)

# 3. Create FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

def semantic_search(query, k=3):
    query_vector = model.encode([query])
    distances, indices = index.search(np.array(query_vector).astype('float32'), k)

    results = []
    for i in range(k):
        idx = indices[0][i]
        node_id = node_list[idx][0]
        results.append((node_id, texts[idx]))
    return results

# Example Search
query = "companies involved in high-end chip manufacturing"
search_results = semantic_search(query)
print(f"Top results for '{query}':")
for res in search_results:
    print(f"- {res[0]}: {res[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Top results for 'companies involved in high-end chip manufacturing':
- OpenAI: OpenAI is a AI Company. valuation: 80B hq: San Francisco
- TSMC: TSMC is a Infrastructure. type: Semiconductor
- NVIDIA: NVIDIA is a Infrastructure. valuation: 2T hq: Santa Clara


## Expanding the Ecosystem & Graph Analytics
We are now adding a broader range of entities including models, universities, and specific hardware. We also calculate centralities to identify the 'hubs' of the AI world.

In [6]:
# Expand nodes: Models
models = [
    ('GPT-4', 'AI Model', {'developer': 'OpenAI', 'type': 'LLM'}),
    ('Claude 3', 'AI Model', {'developer': 'Anthropic', 'type': 'LLM'}),
    ('Llama 3', 'AI Model', {'developer': 'Meta AI', 'type': 'Open Weights'}),
    ('Gemini', 'AI Model', {'developer': 'Google DeepMind', 'type': 'Multimodal'}),
    ('Stable Diffusion', 'AI Model', {'developer': 'Stability AI', 'type': 'Image Gen'})
]

# Expand nodes: Research & Investors
institutions = [
    ('Stanford HAI', 'Research', {'focus': 'Human-Centered AI'}),
    ('MIT CSAIL', 'Research', {'focus': 'Computer Science'}),
    ('Sequoia Capital', 'Investor', {'type': 'VC'}),
    ('Andreessen Horowitz', 'Investor', {'type': 'VC'})
]

for name, cat, attrs in models + institutions:
    explorer.add_node(name, name, cat, **attrs)

# Add complex relationships
extra_rels = [
    ('OpenAI', 'GPT-4', 'Developed By', 5.0),
    ('Anthropic', 'Claude 3', 'Developed By', 5.0),
    ('Meta AI', 'Llama 3', 'Developed By', 5.0),
    ('Google DeepMind', 'Gemini', 'Developed By', 5.0),
    ('Sequoia Capital', 'OpenAI', 'Investment', 4.0),
    ('Andreessen Horowitz', 'Mistral AI', 'Investment', 4.0),
    ('Stanford HAI', 'Google DeepMind', 'Collaboration', 3.0),
    ('MIT CSAIL', 'NVIDIA', 'Research Partner', 3.0)
]

for u, v, rel, w in extra_rels:
    # Add missing nodes if they appear in relationships but not in nodes list
    if u not in explorer.G: explorer.add_node(u, u, 'Misc')
    if v not in explorer.G: explorer.add_node(v, v, 'Misc')
    explorer.add_edge(u, v, rel, weight=w)

# Calculate Graph Analytics
explorer.pagerank = nx.pagerank(explorer.G, weight='weight')
explorer.centrality = nx.betweenness_centrality(explorer.G)

# Update nodes with importance scores
for node in explorer.G.nodes():
    explorer.G.nodes[node]['value'] = explorer.pagerank.get(node, 0.1) * 100

print(f'Expanded to {explorer.G.number_of_nodes()} nodes. Importance scores calculated.')

Expanded to 18 nodes. Importance scores calculated.


In [7]:
def visualize_enhanced_network(eg):
    net = Network(height='700px', width='100%', bgcolor='#0b0d17', font_color='white', directed=True, notebook=True)
    net.set_options('{"physics": {"forceAtlas2Based": {"gravitationalConstant": -50, "centralGravity": 0.01, "springLength": 100, "springConstant": 0.08}, "minVelocity": 0.75, "solver": "forceAtlas2Based"}}')

    color_map = {
        'AI Company': '#00d2ff',
        'Infrastructure': '#ff4b2b',
        'Investor/Cloud': '#ffb347',
        'Research': '#a8ff78',
        'AI Model': '#9d50bb'
    }

    for node, data in eg.G.nodes(data=True):
        color = color_map.get(data.get('category'), '#ffffff')
        # Size is determined by the 'value' (PageRank) calculated earlier
        size = data.get('value', 10)
        net.add_node(node, label=data.get('label'), color=color, size=size, title=f"Importance: {data.get('value', 0):.2f}")

    for u, v, data in eg.G.edges(data=True):
        net.add_edge(u, v, title=data.get('title'), width=data.get('weight')/2)

    net.show('enhanced_ecosystem.html')
    return HTML(filename='enhanced_ecosystem.html')

visualize_enhanced_network(explorer)

enhanced_ecosystem.html
